# Rechercher une meilleure formule de routage TR-HASH

Ce notebook autonome montre tout le code PyTorch et entraîne quatre modèles depuis zéro sur les mêmes données. **Exécuter tout lance 28 entraînements : quatre modèles × sept seeds.** Le notebook de qualification reste séparé.


## 1. Régler l’expérience

`FORMULE_TETES="modulo4"` essaie `a=t%4`. `"quotient12"` reprend `a=(t//12)%4`. Dans les deux cas, la seconde tête vaut `(a+1)%4`. Cette seule formule change pour TR-HASH et son Dense apparié ; les autres routes restent identiques.

Torch 2.13.0, CPU float32, quatre threads. Pour relire sans entraîner, mettre `RUN_RESEARCH=False` et renseigner `RUN_DIRECTORY`. Les données publiques sont téléchargées seulement si elles manquent dans le cache.


In [ ]:
# research_cell:r-config
from types import SimpleNamespace
import random
import torch
from torch import nn
import torch.nn.functional as F

MARGE = 0.001
METRIQUES = ("lm_pretrain", "lm_refinement", "lm_final", "reponses_finales", "oubli")
SEED_DONNEES = 42
config = SimpleNamespace(
    vocab_size=256,
    hidden_size=64,
    num_hidden_layers=2,
    num_attention_heads=4,
    num_key_value_heads=1,
    max_position_embeddings=256,
    num_experts=4,
    top_k=2,
)
protocole = {
    "pretraining": {"epochs": 2, "lr": 0.001},
    "refinement": {"epochs": 1, "lr": 0.0005},
    "sft": {"epochs": 12, "lr": 0.0002},
}
options_adamw = {"betas": (0.9, 0.999), "eps": 1e-08, "weight_decay": 0.01}
VARIANTES = {
    "TR composé ×2 + têtes routées": {
        "partage": 128,
        "expert": 16,
        "dense": 0,
        "echelle": 2.0,
        "attention_routee": True,
    },
    "Dense": {"partage": 0, "expert": 0, "dense": 192, "echelle": 1.0, "attention_routee": False},
    "Dense + têtes routées": {
        "partage": 0,
        "expert": 0,
        "dense": 192,
        "echelle": 1.0,
        "attention_routee": True,
    },
}
REFERENCE = "TR composé ×2 + têtes routées"


def fixer_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)


def lineaire(entree, sortie, residuel=False):
    couche = nn.Linear(entree, sortie, bias=False)
    std = 0.02 / ((2 * config.num_hidden_layers) ** 0.5 if residuel else 1)
    nn.init.normal_(couche.weight, std=std)
    return couche


class SwiGLU(nn.Module):
    def __init__(self, largeur):
        super().__init__()
        self.gate = lineaire(config.hidden_size, largeur)
        self.up = lineaire(config.hidden_size, largeur)
        self.down = lineaire(largeur, config.hidden_size, residuel=True)

    def forward(self, x, token_ids=None):
        return self.down(F.silu(self.gate(x)) * self.up(x))


SEEDS_DEV = (42, 123, 314, 159, 265, 358, 979)
MODE_CANDIDAT = "v0_entree_canaux_mod4_attention_compose"
MODE_CONTEXTE = "primaire_bit1_contextuel_couche1_seule_attention_compose"
MODE_DENSE_ORIGINAL = "attention_compose_mlp_mod81_un_precedent"
CAS = {
    "tr_candidate": (REFERENCE, MODE_CANDIDAT),
    "dense_basic_e": ("Dense", "mod31"),
    "dense_original_c": ("Dense + têtes routées", MODE_DENSE_ORIGINAL),
    "dense_matched_d": ("Dense + têtes routées", MODE_CANDIDAT),
}
MODES_ROTATION_V0 = {MODE_CANDIDAT: (0, 64, 4, 16)}


# Formule de la deuxième couche : le même choix est appliqué à TR et au Dense D.
FORMULE_TETES = "modulo4"  # "modulo4" : t % 4 ; "quotient12" : (t // 12) % 4.
RUN_RESEARCH = True
ALLOW_DOWNLOAD = True
RUN_DIRECTORY = None  # Pour relire : RUN_RESEARCH=False et dossier existant ici.

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import time

DATA_CACHE = Path.cwd() / "artifacts/tr_hash/data_raw"
ARTIFACTS_ROOT = Path.cwd() / "artifacts/tr_hash/research"
NOTEBOOK_FILE = Path.cwd() / "TR_Hash_Research.ipynb"
SEEDS = (42, 123, 314, 159, 265, 358, 979)
MODEL_NAMES = ("tr_candidate", "dense_basic_e", "dense_original_c", "dense_matched_d")
MODEL_PARAMETERS = {name: 110976 for name in MODEL_NAMES}
STAGES = protocole
if FORMULE_TETES not in {"modulo4", "quotient12"}:
    raise ValueError("FORMULE_TETES doit valoir 'modulo4' ou 'quotient12'.")


## 2. Construire les quatre modèles

Tous ont deux couches de largeur 64, quatre têtes Q, une tête KV et **110 976 paramètres**. La branche TR conserve quatre experts de largeur 16, une branche partagée de largeur 128 et deux routes fixes 50/50 ×2.

| Modèle | Attention | MLP |
|---|---|---|
| TR-HASH candidat | formule choisie et rotation V0 | partagé + experts |
| Dense E basique | toutes les têtes, sans rotation V0 | SwiGLU 192 |
| Dense C original | têtes HeadCompose5 originales, sans rotation V0 | SwiGLU 192 |
| Dense D apparié | mêmes têtes et rotation V0 que TR-HASH | SwiGLU 192 |

Le choix `modulo4` simplifie le sélecteur ; il ne garantit ni moins de collisions ni de meilleurs résultats. Le contrôle C garde toujours son attention originale. Le petit MoE inspiré de Qwen reste dans le notebook de qualification.


In [ ]:
# research_cell:r-model
def choisir_tetes_couche_1(tokens):
    """Deux têtes voisines, choisies à partir de l’ID du token courant."""
    if FORMULE_TETES == "modulo4":
        a = tokens % 4
    elif FORMULE_TETES == "quotient12":
        a = tokens // 12 % 4
    else:
        raise ValueError("Formule de têtes inconnue.")
    return torch.stack((a, (a + 1) % 4), dim=-1)


def decaler_lag(ids, decalage):
    if ids.shape[1] <= decalage:
        return torch.zeros_like(ids)
    return torch.cat((torch.zeros_like(ids[:, :decalage]), ids[:, :-decalage]), dim=1)


def calculer_routes(
    tokens, couche, mode="compose", precedent=None, role="mlp", avant_precedent=None
):
    if role not in {"mlp", "attention"}:
        raise ValueError(f"Rôle de routage inconnu : {role}")
    if mode == MODE_CANDIDAT and role == "attention" and (couche == 1):
        return choisir_tetes_couche_1(tokens)
    if mode == MODE_CANDIDAT:
        mode = MODE_CONTEXTE
    if mode not in {"compose", "mod31", MODE_CONTEXTE, MODE_DENSE_ORIGINAL}:
        raise ValueError(f"Carte de routage inconnue : {mode}")
    a = (tokens + couche) % config.num_experts
    if mode == "compose" or (role == "attention" and mode in {MODE_CONTEXTE, MODE_DENSE_ORIGINAL}):
        pas = 1 + (tokens // config.num_experts + couche) % (config.num_experts - 1)
        return torch.stack((a, (a + pas) % config.num_experts), dim=-1)
    if precedent is None:
        precedent = torch.zeros_like(tokens)
    if mode == "mod31":
        contexte = precedent % 31
    elif mode == MODE_DENSE_ORIGINAL:
        contexte = precedent % 81 // 3**couche % 3
    else:
        if avant_precedent is None:
            avant_precedent = torch.zeros_like(precedent)
        u = (precedent + 5 * avant_precedent) % 9
        if couche == 1:
            a = (tokens + 1) % 4 ^ 2 * (u % 3 % 2)
            pas = 1 + (tokens // 4 + 2 * (u // 3) + 1) % 3
            return torch.stack((a, (a + pas) % 4), dim=-1)
        contexte = u % 3
    terme = (1 + couche % 2) * contexte
    pas = 1 + (tokens // config.num_experts + terme + couche) % (config.num_experts - 1)
    return torch.stack((a, (a + pas) % config.num_experts), dim=-1)


def table_routes(couche, mode="compose", nombre=None, role="mlp"):
    tokens = torch.arange(config.vocab_size if nombre is None else nombre)
    return calculer_routes(tokens, couche, mode, role=role)


def choisir_routes(ids, couche, mode, table, role="mlp"):
    if mode == "compose" or (
        role == "attention" and mode in {MODE_CANDIDAT, MODE_CONTEXTE, MODE_DENSE_ORIGINAL}
    ):
        return table[ids]
    p1 = decaler_lag(ids, 1)
    p2 = decaler_lag(ids, 2) if mode in {MODE_CANDIDAT, MODE_CONTEXTE} else None
    return calculer_routes(ids, couche, mode, p1, role, p2)


def indices_canaux_v0(token_ids):
    if token_ids.ndim != 2 or token_ids.dtype != torch.long:
        raise ValueError("B x T original token IDs must be torch.long")
    channels = torch.arange(64, dtype=torch.long, device=token_ids.device).view(1, 1, 64)
    return (channels + 16 * (token_ids % 4).unsqueeze(-1)) % 64


def lier_entree_valeur(x, token_ids, couche, mode):
    if couche != 0 or mode not in MODES_ROTATION_V0:
        return x
    if (
        x.ndim != 3
        or x.shape[-1] != 64
        or token_ids.shape != x.shape[:2]
        or (token_ids.device != x.device)
    ):
        raise ValueError("Expected B x T x64 and same-device original B x T IDs")
    return x.gather(-1, indices_canaux_v0(token_ids))


def rope(x):
    longueur, dimension = x.shape[-2:]
    positions = torch.arange(longueur, device=x.device)[:, None]
    frequences = 10000.0 ** (-torch.arange(0, dimension, 2, device=x.device) / dimension)
    angles = positions * frequences
    angles = torch.cat((angles, angles), dim=-1)
    gauche, droite = x.chunk(2, dim=-1)
    return x * angles.cos() + torch.cat((-droite, gauche), dim=-1) * angles.sin()


class TRHashMLP(nn.Module):
    def __init__(self, couche, partage, expert, echelle, routage="compose", poids_primaire=0.5):
        super().__init__()
        self.shared = SwiGLU(partage)
        self.experts = nn.ModuleList([SwiGLU(expert) for _ in range(config.num_experts)])
        self.echelle = echelle
        if not 0 <= poids_primaire <= 1:
            raise ValueError("poids_primaire doit être entre 0 et 1")
        self.poids_primaire = poids_primaire
        self.normalisation_fusion = (0.5 / (poids_primaire**2 + (1 - poids_primaire) ** 2)) ** 0.5
        self.couche, self.routage = (couche, routage)
        self.register_buffer("routes", table_routes(couche, routage, role="mlp"))

    def forward(self, x, token_ids):
        routes = choisir_routes(token_ids, self.couche, self.routage, self.routes, role="mlp")
        routee = torch.zeros_like(x)
        for numero, expert in enumerate(self.experts):
            primaire = (routes[..., :1] == numero).to(x.dtype)
            secondaire = (routes[..., 1:2] == numero).to(x.dtype)
            poids = self.normalisation_fusion * (
                self.poids_primaire * primaire + (1 - self.poids_primaire) * secondaire
            )
            routee = routee + poids * expert(x)
        return self.shared(x) + self.echelle * routee


class AttentionCausale(nn.Module):
    def __init__(self, couche, attention_routee=False, routage="compose"):
        super().__init__()
        self.attention_routee = attention_routee
        self.couche, self.routage = (couche, routage)
        if attention_routee:
            assert config.num_attention_heads == config.num_experts
            self.register_buffer("routes_tetes", table_routes(couche, routage, role="attention"))
        self.h = config.num_attention_heads
        self.kv = config.num_key_value_heads
        self.d = config.hidden_size // self.h
        self.q = lineaire(config.hidden_size, self.h * self.d)
        self.k = lineaire(config.hidden_size, self.kv * self.d)
        self.v = lineaire(config.hidden_size, self.kv * self.d)
        self.o = lineaire(config.hidden_size, config.hidden_size, residuel=True)
        self.norm_q = nn.RMSNorm(self.d, eps=1e-06)
        self.norm_k = nn.RMSNorm(self.d, eps=1e-06)

    def forward(self, x, token_ids):
        b, t, _ = x.shape
        q = self.q(x).view(b, t, self.h, self.d).transpose(1, 2)
        k = self.k(x).view(b, t, self.kv, self.d).transpose(1, 2)
        v = (
            self.v(lier_entree_valeur(x, token_ids, self.couche, self.routage))
            .view(b, t, self.kv, self.d)
            .transpose(1, 2)
        )
        q, k = (rope(self.norm_q(q)), rope(self.norm_k(k)))
        k = k.repeat_interleave(self.h // self.kv, dim=1)
        v = v.repeat_interleave(self.h // self.kv, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=0.0)
        if self.attention_routee:
            selection = F.one_hot(
                choisir_routes(
                    token_ids, self.couche, self.routage, self.routes_tetes, role="attention"
                ),
                num_classes=self.h,
            ).sum(dim=-2)
            masque = selection.permute(0, 2, 1).unsqueeze(-1).to(y.dtype)
            y = y * masque * (self.h / config.top_k) ** 0.5
        return self.o(y.transpose(1, 2).reshape(b, t, config.hidden_size))


class Bloc(nn.Module):
    def __init__(self, mlp, couche, attention_routee, routage):
        super().__init__()
        self.attention = AttentionCausale(couche, attention_routee, routage)
        self.norm1 = nn.RMSNorm(config.hidden_size, eps=1e-06)
        self.norm2 = nn.RMSNorm(config.hidden_size, eps=1e-06)
        self.mlp = mlp

    def forward(self, x, ids):
        x = x + self.attention(self.norm1(x), ids)
        return x + self.mlp(self.norm2(x), ids)


class ModeleLangage(nn.Module):
    def __init__(self, variante, routage="compose", poids_primaire=0.5):
        super().__init__()
        profil = VARIANTES[variante]
        self.embedding = nn.Embedding(config.vocab_size, config.hidden_size)
        nn.init.normal_(self.embedding.weight, std=0.02)
        self.layers = nn.ModuleList(
            [
                Bloc(
                    SwiGLU(profil["dense"])
                    if profil["dense"]
                    else TRHashMLP(
                        i,
                        profil["partage"],
                        profil["expert"],
                        profil["echelle"],
                        routage,
                        poids_primaire,
                    ),
                    i,
                    profil["attention_routee"],
                    routage,
                )
                for i in range(config.num_hidden_layers)
            ]
        )
        self.norm = nn.RMSNorm(config.hidden_size, eps=1e-06)

    def forward(self, ids):
        if ids.shape[1] > config.max_position_embeddings:
            raise ValueError(
                f"L'entrée dépasse {config.max_position_embeddings} tokens de contexte."
            )
        x = self.embedding(ids)
        for couche in self.layers:
            x = couche(x, ids)
        return F.linear(self.norm(x), self.embedding.weight)

    def num_parameters(self):
        return sum((p.numel() for p in self.parameters()))


def initialiser_modeles(seed, noms=None, routage="compose", poids_primaire=0.5):
    fixer_seed(seed)
    reference = ModeleLangage(REFERENCE, routage, poids_primaire)
    communs = {
        nom: poids.clone() for nom, poids in reference.named_parameters() if ".mlp." not in nom
    }
    modeles = {}
    for nom in list(VARIANTES) if noms is None else noms:
        fixer_seed(seed)
        modele = ModeleLangage(nom, routage, poids_primaire)
        modele.load_state_dict(communs, strict=False)
        modeles[nom] = modele
    return modeles


def initialiser_cas(seed, cas="tr_candidate"):
    variante, mode = CAS[cas]
    return initialiser_modeles(seed, [variante], mode, 0.5)[variante]


def build_model(seed, name):
    """Initialise un modèle neuf, sans charger de checkpoint."""
    if name not in MODEL_NAMES:
        raise ValueError(f"Modèle inconnu : {name}")
    model = initialiser_cas(seed, name)
    if model.num_parameters() != MODEL_PARAMETERS[name]:
        raise ValueError(f"Budget de paramètres inattendu pour {name}.")
    return model


## 3. Préparer les données

TinyShakespeare fournit 2 048 blocs d’entraînement et 256 de validation ; Alpaca fournit 512 réponses d’entraînement et 128 de validation. Le prompt et le remplissage sont masqués dans la perte. Les lots, leur ordre et les empreintes SHA-256 restent identiques entre modèles. Aucun texte de préentraînement n’est ajouté au SFT.


In [ ]:
# research_cell:r-data
def preparer_donnees_dev():
    import hashlib
    import json
    from urllib.request import urlopen
    from torch.nn.utils.rnn import pad_sequence

    TAILLE_LOT = 16
    LONGUEUR_BLOC = 128
    NB_PRETRAIN, NB_VALIDATION = 2048, 256
    NB_SFT, NB_SFT_VALIDATION = 512, 128
    cache = DATA_CACHE
    cache.mkdir(parents=True, exist_ok=True)

    def charger(nom, url, sha256):
        fichier = cache / nom
        if fichier.exists():
            contenu = fichier.read_bytes()
        else:
            if not ALLOW_DOWNLOAD:
                raise FileNotFoundError(f"Donnée absente du cache : {fichier}")
            print(f"Téléchargement : {nom}", flush=True)
            with urlopen(url, timeout=90) as reponse:
                contenu = reponse.read()
        if hashlib.sha256(contenu).hexdigest() != sha256:
            raise ValueError(f"Empreinte SHA-256 incorrecte : {fichier}")
        if not fichier.exists():
            fichier.write_bytes(contenu)
        return contenu

    texte = charger(
        "tinyshakespeare.txt",
        "https://raw.githubusercontent.com/karpathy/char-rnn/370cbcd448eb7daf32f21a6be560b70e0b33c4e3/data/tinyshakespeare/input.txt",
        "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed",
    )
    alpaca = json.loads(
        charger(
            "alpaca_data.json",
            "https://raw.githubusercontent.com/tatsu-lab/stanford_alpaca/f13496211289def0ff88ae673389ae14a818b4b3/alpaca_data.json",
            "2eddafc6b977608d778aaab8dfc7e50e547b3af9826dfb9e909d9fc362e4a419",
        )
    )

    def encoder(texte):
        return torch.tensor([list(texte.encode("utf-8"))], dtype=torch.long)

    def preparer_lm(octets, nombre):
        ids = torch.tensor(list(octets), dtype=torch.long)
        blocs = ids.unfold(0, LONGUEUR_BLOC + 1, LONGUEUR_BLOC)
        assert len(blocs) >= nombre, "Pas assez de texte pour ce nombre de blocs."
        ordre = torch.randperm(len(blocs), generator=torch.Generator().manual_seed(SEED_DONNEES))[
            :nombre
        ]
        return [(bloc[None, :], bloc[None, :].clone()) for bloc in blocs[ordre]]

    def preparer_sft(paires):
        exemples = []
        for instruction, reponse in paires:
            prompt = f"Instruction: {instruction}\nAnswer: "
            ids = encoder(prompt + reponse + "\n")
            labels = ids.clone()
            labels[:, : len(prompt.encode("utf-8"))] = -100
            exemples.append((ids, labels))
        return exemples

    def lots(exemples):
        resultat = []
        for debut in range(0, len(exemples), TAILLE_LOT):
            groupe = exemples[debut : debut + TAILLE_LOT]
            ids = pad_sequence([x[0] for x, y in groupe], batch_first=True, padding_value=0)
            labels = pad_sequence([y[0] for x, y in groupe], batch_first=True, padding_value=-100)
            resultat.append((ids, labels))
        return resultat

    separation = int(0.9 * len(texte))
    exemples_pretrain = preparer_lm(texte[:separation], NB_PRETRAIN)
    exemples_validation = preparer_lm(texte[separation:], NB_VALIDATION)
    donnees_pretrain = lots(exemples_pretrain)
    validation_texte = lots(exemples_validation)
    eligibles, deja_vues = [], set()
    for ligne in alpaca:
        instruction, reponse = ligne["instruction"].strip(), ligne["output"].strip()
        cle = " ".join(instruction.casefold().split())
        if ligne["input"].strip() or not instruction or not reponse or cle in deja_vues:
            continue
        complet = f"Instruction: {instruction}\nAnswer: {reponse}\n"
        if len(complet.encode("utf-8")) > config.max_position_embeddings:
            continue
        deja_vues.add(cle)
        eligibles.append((instruction, reponse))
    random.Random(SEED_DONNEES).shuffle(eligibles)
    questions_reponses = eligibles[:NB_SFT]
    questions_validation = eligibles[NB_SFT : NB_SFT + NB_SFT_VALIDATION]
    donnees_sft = lots(preparer_sft(questions_reponses))
    validation_sft = lots(preparer_sft(questions_validation))

    def decrire_lots(liste):
        empreinte = hashlib.sha256()
        for ids, labels in liste:
            empreinte.update(str(tuple(ids.shape)).encode())
            empreinte.update(bytes(ids.flatten().tolist()))
            empreinte.update(bytes((labels != -100).flatten().tolist()))
        return {
            "sha256_lots": empreinte.hexdigest(),
            "exemples": sum(ids.shape[0] for ids, labels in liste),
            "lots": len(liste),
            "tokens_supervises": sum((labels[:, 1:] != -100).sum().item() for ids, labels in liste),
        }

    donnees = {"pretraining": donnees_pretrain, "refinement": donnees_pretrain, "sft": donnees_sft}
    partitions = {
        "train_lm": donnees_pretrain,
        "validation_lm": validation_texte,
        "train_sft": donnees_sft,
        "validation_sft": validation_sft,
    }
    manifest_donnees = {nom: decrire_lots(liste) for nom, liste in partitions.items()}
    MANIFEST_ATTENDU = {
        "train_lm": {
            "sha256_lots": "d4b214ddaf2d45923b2210c106fa759294be0cebb2ec95500c6ae62b2879901d",
            "exemples": 2048,
            "lots": 128,
            "tokens_supervises": 262144,
        },
        "validation_lm": {
            "sha256_lots": "cb1c926598949d73e87e557d2dbdffe99491aad185a9ba2617ba00120ce722f5",
            "exemples": 256,
            "lots": 16,
            "tokens_supervises": 32768,
        },
        "train_sft": {
            "sha256_lots": "26f2e9c9e75685046bd3217752c08e908b81386b7190b5aa6a4d45824c43714e",
            "exemples": 512,
            "lots": 32,
            "tokens_supervises": 37691,
        },
        "validation_sft": {
            "sha256_lots": "5f017967e60384685a7526311794375dc2eb9e673ab0a4ecd908efdd9215bda9",
            "exemples": 128,
            "lots": 8,
            "tokens_supervises": 9386,
        },
    }
    assert manifest_donnees == MANIFEST_ATTENDU
    budget_updates = sum(
        regles["epochs"] * len(donnees[etape]) for etape, regles in protocole.items()
    )
    budget_tokens = sum(
        regles["epochs"] * decrire_lots(donnees[etape])["tokens_supervises"]
        for etape, regles in protocole.items()
    )
    assert budget_updates == 768 and budget_tokens == 1238724
    return {
        "donnees": donnees,
        "validation_texte": validation_texte,
        "validation_sft": validation_sft,
        "manifest": manifest_donnees,
        "budget_updates": budget_updates,
        "budget_tokens": budget_tokens,
    }


RAW_DATA = {
    "tinyshakespeare.txt": {
        "url": "https://raw.githubusercontent.com/karpathy/char-rnn/370cbcd448eb7daf32f21a6be560b70e0b33c4e3/data/tinyshakespeare/input.txt",
        "sha256": "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed",
    },
    "alpaca_data.json": {
        "url": "https://raw.githubusercontent.com/tatsu-lab/stanford_alpaca/f13496211289def0ff88ae673389ae14a818b4b3/alpaca_data.json",
        "sha256": "2eddafc6b977608d778aaab8dfc7e50e547b3af9826dfb9e909d9fc362e4a419",
    },
}

DATA_MANIFEST = {
    "train_lm": {
        "sha256_lots": "d4b214ddaf2d45923b2210c106fa759294be0cebb2ec95500c6ae62b2879901d",
        "exemples": 2048,
        "lots": 128,
        "tokens_supervises": 262144,
    },
    "validation_lm": {
        "sha256_lots": "cb1c926598949d73e87e557d2dbdffe99491aad185a9ba2617ba00120ce722f5",
        "exemples": 256,
        "lots": 16,
        "tokens_supervises": 32768,
    },
    "train_sft": {
        "sha256_lots": "26f2e9c9e75685046bd3217752c08e908b81386b7190b5aa6a4d45824c43714e",
        "exemples": 512,
        "lots": 32,
        "tokens_supervises": 37691,
    },
    "validation_sft": {
        "sha256_lots": "5f017967e60384685a7526311794375dc2eb9e673ab0a4ecd908efdd9215bda9",
        "exemples": 128,
        "lots": 8,
        "tokens_supervises": 9386,
    },
}


## 4. Définir l’entraînement

Chaque modèle suit deux époques de préentraînement texte, une de raffinement et douze de SFT, avec un nouvel AdamW à chaque étape. Le budget total est de **21 504 mises à jour et 34 684 272 cibles supervisées**.

Le code et la formule sont sauvegardés avant le premier entraînement. Chaque étape conserve ses poids et mesure la CE texte et la CE des réponses. Tous les entraînements se terminent avant la comparaison.


In [ ]:
# research_cell:r-training
def calculer_loss(modele, input_ids, labels, reduction="mean"):
    logits = modele(input_ids)
    return F.cross_entropy(
        logits[:, :-1].reshape(-1, config.vocab_size),
        labels[:, 1:].reshape(-1),
        ignore_index=-100,
        reduction=reduction,
    )


def faire_epoque(modele, exemples, optimizer):
    modele.train()
    total, tokens = (0.0, 0)
    for input_ids, labels in exemples:
        optimizer.zero_grad(set_to_none=True)
        loss = calculer_loss(modele, input_ids, labels)
        loss.backward()
        optimizer.step()
        nombre = (labels[:, 1:] != -100).sum().item()
        total += loss.item() * nombre
        tokens += nombre
    modele.eval()
    return (total / tokens, len(exemples), tokens)


def evaluer(modele, exemples):
    modele.eval()
    total, tokens = (0.0, 0)
    with torch.inference_mode():
        for input_ids, labels in exemples:
            total += calculer_loss(modele, input_ids, labels, reduction="sum").item()
            tokens += (labels[:, 1:] != -100).sum().item()
    return total / tokens


def _json_bytes(value):
    return (
        json.dumps(value, ensure_ascii=False, sort_keys=True, allow_nan=False, indent=2) + "\n"
    ).encode()


def _atomic_json(path, value):
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("wb") as out:
        out.write(_json_bytes(value))
        out.flush()
        os.fsync(out.fileno())
    os.replace(temporary, path)


def _empreinte(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def _event(run_dir, event):
    item = {"utc": datetime.now(timezone.utc).isoformat(), **event}
    with (run_dir / "progress.jsonl").open("a", encoding="utf-8") as out:
        out.write(json.dumps(item, ensure_ascii=False, allow_nan=False) + "\n")
        out.flush()
        os.fsync(out.fileno())
    if event["event"] == "epoch_complete":
        print(
            f"{event['model']} · seed {event['seed']} · {event['stage']} {event['epoch']} : CE {event['train_ce']:.5f}",
            flush=True,
        )
    elif event["event"] == "stage_complete":
        print(
            f"  Validation : texte {event['lm_ce']:.5f} · réponses {event['response_ce']:.5f}",
            flush=True,
        )
    elif event["event"] == "model_started":
        print(f"\n{event['model']} · seed {event['seed']}", flush=True)
    elif event["event"] == "completed":
        print("Les 28 entraînements sont terminés.", flush=True)


def _capturer_sources():
    """Conserve le notebook et les cellules réellement exécutées quand Jupyter les fournit."""
    saved = json.loads(NOTEBOOK_FILE.read_text()) if NOTEBOOK_FILE.is_file() else None
    sources = (
        {
            cell["id"]: "".join(cell["source"])
            for cell in saved["cells"]
            if cell["cell_type"] == "code"
        }
        if saved
        else {}
    )
    history = globals().get("In", [])
    for ident in ("r-config", "r-model", "r-data", "r-training", "r-plots", "r-run"):
        matches = [
            text
            for text in history
            if isinstance(text, str) and text.startswith("# research_cell:" + ident + "\n")
        ]
        if matches:
            sources[ident] = matches[-1]
    required = {"r-config", "r-model", "r-data", "r-training", "r-plots", "r-run"}
    if not required.issubset(sources):
        raise ValueError(
            "Enregistrer le notebook ou exécuter ses cellules dans l’ordre avant de lancer."
        )
    if saved:
        for cell in saved["cells"]:
            if cell["cell_type"] == "code":
                cell["outputs"], cell["execution_count"] = [], None
                if cell["id"] in sources:
                    cell["source"] = sources[cell["id"]].splitlines(keepends=True)
    return saved, sources


def build_protocol(source_hashes):
    return {
        "experiment": "tr_hash_research",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "head1_formula": FORMULE_TETES,
        "seeds": list(SEEDS),
        "models": list(MODEL_NAMES),
        "num_parameters": MODEL_PARAMETERS.copy(),
        "architecture": {
            "hidden": 64,
            "layers": 2,
            "query_heads": 4,
            "kv_heads": 1,
            "shared_width": 128,
            "experts": 4,
            "expert_width": 16,
            "top_k": 2,
            "fixed_weights": [0.5, 0.5],
            "routed_scale": 2,
            "dense_width": 192,
        },
        "attention": {
            "tr_candidate": "formule choisie + V0",
            "dense_basic_e": "toutes les têtes, V0 identité",
            "dense_original_c": "HeadCompose5 original, V0 identité",
            "dense_matched_d": "mêmes têtes et V0 que TR",
        },
        "stages": STAGES,
        "optimizer": {
            "name": "AdamW",
            "fresh_each_stage": True,
            "betas": [0.9, 0.999],
            "eps": 1e-8,
            "weight_decay": 0.01,
            "clipping": None,
        },
        "data": {
            "seed": 42,
            "raw": RAW_DATA,
            "manifest": DATA_MANIFEST,
            "batch_order": "fixe, sans mélange entre époques",
            "test_partition": None,
        },
        "runtime": {
            "torch": str(torch.__version__),
            "device": "cpu",
            "dtype": "float32",
            "threads": 4,
            "deterministic_algorithms": True,
            "matmul_precision": "highest",
        },
        "budget": {
            "trajectories": 28,
            "total_updates": 21504,
            "total_supervised_targets": 34684272,
            "fresh_optimizers": 84,
            "validation_passes": 168,
            "checkpoints": 84,
        },
        "metrics": {
            "all_lower_is_better": True,
            "oubli": "lm_final-lm_refinement",
            "win": "TR - contrôle <= -0.001",
        },
        "source_cell_sha256": source_hashes,
        "interpretation": "sept seeds de développement ; aucune confirmation indépendante ni promotion automatique",
    }


def prepare_run():
    notebook, sources = _capturer_sources()
    run_dir = ARTIFACTS_ROOT / datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%S_%fZ")
    run_dir.mkdir(parents=True, exist_ok=False)
    source_dir = run_dir / "Source"
    source_dir.mkdir()
    _atomic_json(source_dir / "cells.json", sources)
    if notebook:
        _atomic_json(source_dir / "notebook.ipynb", notebook)
    hashes = {ident: hashlib.sha256(code.encode()).hexdigest() for ident, code in sources.items()}
    protocol = build_protocol(hashes)
    protocol["source_json_sha256"] = _empreinte(source_dir / "cells.json")
    protocol["protocol_sha256"] = hashlib.sha256(_json_bytes(protocol)).hexdigest()
    _atomic_json(run_dir / "protocol.json", protocol)
    return run_dir


def load_saved_results(run_dir):
    result = json.loads((Path(run_dir) / "results.json").read_text())
    expected = {(name, seed) for name in MODEL_NAMES for seed in SEEDS}
    observed = {(row["model"], row["seed"]) for row in result.get("trajectories", [])}
    if (
        result.get("status") != "complete"
        or result["protocol"].get("experiment") != "tr_hash_research"
        or len(result["trajectories"]) != 28
        or observed != expected
    ):
        raise ValueError("Les 28 entraînements complets de cette expérience sont requis.")
    return result


def run_research(run_dir):
    run_dir = Path(run_dir).resolve()
    protocol = json.loads((run_dir / "protocol.json").read_text())
    unsigned = {k: v for k, v in protocol.items() if k != "protocol_sha256"}
    if hashlib.sha256(_json_bytes(unsigned)).hexdigest() != protocol["protocol_sha256"]:
        raise ValueError("Le protocole a changé après sa sauvegarde.")
    if FORMULE_TETES != protocol["head1_formula"]:
        raise ValueError("La formule a changé entre la préparation et le lancement.")
    if _empreinte(run_dir / "Source/cells.json") != protocol["source_json_sha256"]:
        raise ValueError("Le code sauvegardé a changé après la préparation.")
    if str(torch.__version__).split("+")[0] != "2.13.0":
        raise RuntimeError("Cette expérience utilise Torch 2.13.0.")
    with (run_dir / "run.lock").open("x") as out:
        out.write(f"pid={os.getpid()}\n")
    torch.set_default_device("cpu")
    torch.set_default_dtype(torch.float32)
    torch.set_num_threads(4)
    torch.set_float32_matmul_precision("highest")
    torch.use_deterministic_algorithms(True)
    totals = {
        "trajectories": 0,
        "total_updates": 0,
        "total_supervised_targets": 0,
        "fresh_optimizers": 0,
        "validation_passes": 0,
        "checkpoints": 0,
    }
    completed = []
    started = time.monotonic()
    try:
        data = preparer_donnees_dev()
        if data["manifest"] != DATA_MANIFEST:
            raise ValueError("Les empreintes des lots diffèrent du protocole.")
        _atomic_json(run_dir / "data_manifest.json", data["manifest"])
        for seed in SEEDS:
            for name in MODEL_NAMES:
                if FORMULE_TETES != protocol["head1_formula"]:
                    raise ValueError("La formule doit rester identique pendant l’expérience.")
                model = build_model(seed, name)
                directory = run_dir / "trajectories" / f"seed{seed}_{name}"
                directory.mkdir(parents=True, exist_ok=False)
                row = {
                    "seed": seed,
                    "model": name,
                    "num_parameters": model.num_parameters(),
                    "stage_points": {},
                    "epochs": [],
                    "checkpoints": {},
                }
                _event(run_dir, {"event": "model_started", "seed": seed, "model": name})
                for stage, settings in STAGES.items():
                    optimizer = torch.optim.AdamW(
                        model.parameters(), lr=settings["lr"], **options_adamw
                    )
                    totals["fresh_optimizers"] += 1
                    for epoch in range(1, settings["epochs"] + 1):
                        ce, updates, targets = faire_epoque(
                            model, data["donnees"][stage], optimizer
                        )
                        row["epochs"].append(
                            {
                                "stage": stage,
                                "epoch": epoch,
                                "train_ce": ce,
                                "updates": updates,
                                "supervised_targets": targets,
                            }
                        )
                        totals["total_updates"] += updates
                        totals["total_supervised_targets"] += targets
                        _atomic_json(directory / "progress.json", row)
                        _event(
                            run_dir,
                            {
                                "event": "epoch_complete",
                                "seed": seed,
                                "model": name,
                                **row["epochs"][-1],
                            },
                        )
                    point = {
                        "lm_ce": evaluer(model, data["validation_texte"]),
                        "response_ce": evaluer(model, data["validation_sft"]),
                    }
                    row["stage_points"][stage] = point
                    totals["validation_passes"] += 2
                    checkpoint = directory / f"{stage}.pt"
                    with checkpoint.open("xb") as out:
                        torch.save(model.state_dict(), out)
                        out.flush()
                        os.fsync(out.fileno())
                    row["checkpoints"][stage] = {
                        "path": str(checkpoint.relative_to(run_dir)),
                        "sha256": _empreinte(checkpoint),
                    }
                    totals["checkpoints"] += 1
                    _atomic_json(directory / "progress.json", row)
                    _event(
                        run_dir,
                        {
                            "event": "stage_complete",
                            "seed": seed,
                            "model": name,
                            "stage": stage,
                            **point,
                        },
                    )
                points = row["stage_points"]
                row["metrics"] = {
                    "lm_pretrain": points["pretraining"]["lm_ce"],
                    "lm_refinement": points["refinement"]["lm_ce"],
                    "lm_final": points["sft"]["lm_ce"],
                    "reponses_finales": points["sft"]["response_ce"],
                    "oubli": points["sft"]["lm_ce"] - points["refinement"]["lm_ce"],
                }
                _atomic_json(directory / "metrics.json", row)
                completed.append(row)
                totals["trajectories"] += 1
                _atomic_json(
                    run_dir / "partial_results.json",
                    {
                        "status": "incomplete",
                        "protocol": protocol,
                        "trajectories": completed,
                        "actual_budget": totals,
                    },
                )
                del model, optimizer
        if totals != protocol["budget"]:
            raise ValueError(f"Le budget final diffère du protocole : {totals}")
        result = {
            "status": "complete",
            "protocol": protocol,
            "trajectories": completed,
            "actual_budget": totals,
            "elapsed_seconds": time.monotonic() - started,
        }
        _atomic_json(run_dir / "results.json", result)
        _event(run_dir, {"event": "completed", **totals})
        return load_saved_results(run_dir)
    except Exception as error:
        _event(
            run_dir,
            {"event": "failed", "error": repr(error), "completed_trajectories": len(completed)},
        )
        raise


def comparer_resultats(result):
    """Garde toutes les seeds et les cinq métriques, avec la même marge partout."""
    rows = result["trajectories"]
    indexed = {(row["model"], row["seed"]): row["metrics"] for row in rows}
    grid = []
    for opponent in MODEL_NAMES[1:]:
        for seed in SEEDS:
            for metric in METRIQUES:
                delta = indexed["tr_candidate", seed][metric] - indexed[opponent, seed][metric]
                grid.append(
                    {
                        "controle": opponent,
                        "seed": seed,
                        "metrique": metric,
                        "ecart_TR_moins_controle": delta,
                        "victoire": delta <= -MARGE,
                    }
                )
    scores = {
        name: sum(row["victoire"] for row in grid if row["controle"] == name)
        for name in MODEL_NAMES[1:]
    }
    return {
        "marge": MARGE,
        "scores_sur_35": scores,
        "comparaison_E_C_sur_70": scores["dense_basic_e"] + scores["dense_original_c"],
        "total_sur_105": sum(scores.values()),
        "toutes_victoires_developpement": all(row["victoire"] for row in grid),
        "grille_complete": grid,
        "confirmation_independante": False,
    }


## 5. Préparer les graphiques

Plus la CE est faible, meilleur est le résultat. Les courbes texte utilisent trois validations réellement mesurées ; les douze points SFT montrent la CE d’entraînement. L’oubli vaut `CE texte finale − CE texte après raffinement`. Les points des sept seeds restent visibles ; les figures sont exportées en PNG 400 dpi et SVG.


In [ ]:
# research_cell:r-plots
"Figures calculées à partir des 28 entraînements sauvegardés."

import math
from statistics import mean

SEEDS = (42, 123, 314, 159, 265, 358, 979)
MODELS = ("tr_candidate", "dense_basic_e", "dense_original_c", "dense_matched_d")
LABELS = {
    "tr_candidate": "TR-HASH candidat",
    "dense_basic_e": "Dense E — attention simple",
    "dense_matched_d": "Dense D — mêmes têtes et V0",
    "dense_original_c": "Dense C — têtes originales",
}
COLORS = {
    "tr_candidate": "#2166ac",
    "dense_basic_e": "#60666f",
    "dense_original_c": "#d97706",
    "dense_matched_d": "#16845b",
}
METRICS = ("lm_pretrain", "lm_refinement", "lm_final", "reponses_finales", "oubli")
METRIC_LABELS = (
    "Texte après préentraînement",
    "Texte après raffinement",
    "Texte après SFT",
    "Réponses après SFT",
    "Oubli",
)
PLOT_STAGES = ("pretraining", "refinement", "sft")
FIGURES = (
    ("lm_checkpoints", "CE texte aux trois checkpoints"),
    ("final_responses", "CE des réponses après SFT"),
    ("sft_training_ce", "CE d’entraînement SFT"),
    ("forgetting", "Oubli sur les sept seeds"),
    ("paired_deltas", "Écarts appariés contre TR-HASH"),
    ("final_text_vs_responses", "Texte final et réponses finales"),
)


def _read_complete_results(run_dir: Path) -> tuple[dict, dict, str]:
    path = run_dir / "results.json"
    raw = path.read_bytes()
    payload = json.loads(raw)
    if payload.get("status") != "complete":
        raise ValueError(
            "Les figures exigent une expérience complète ; aucun choix sur un résultat partiel."
        )
    rows = payload.get("trajectories")
    if not isinstance(rows, list) or len(rows) != 28:
        raise ValueError(
            "Les figures exigent exactement 28 trajectoires, quatre modèles × sept seeds."
        )
    indexed = {}
    for row in rows:
        key = (row.get("model"), row.get("seed"))
        if key in indexed or key[0] not in MODELS or key[1] not in SEEDS:
            raise ValueError("Modèle/seed manquant, dupliqué ou hors protocole dans les résultats.")
        for metric in METRICS:
            value = row.get("metrics", {}).get(metric)
            if not isinstance(value, (int, float)) or not math.isfinite(value):
                raise ValueError(f"Métrique absente ou non finie : {key}, {metric}")
        for stage in PLOT_STAGES:
            point = row.get("stage_points", {}).get(stage, {})
            for metric in ("lm_ce", "response_ce"):
                value = point.get(metric)
                if not isinstance(value, (int, float)) or not math.isfinite(value):
                    raise ValueError(f"Checkpoint incomplet : {key}, {stage}, {metric}")
        epochs = sorted(
            (epoch for epoch in row.get("epochs", []) if epoch.get("stage") == "sft"),
            key=lambda epoch: epoch.get("epoch", -1),
        )
        if [epoch.get("epoch") for epoch in epochs] != list(range(1, 13)):
            raise ValueError(f"Douze CE d’entraînement SFT sont nécessaires : {key}")
        if not all(
            isinstance(epoch.get("train_ce"), (int, float)) and math.isfinite(epoch["train_ce"])
            for epoch in epochs
        ):
            raise ValueError(f"CE d’entraînement SFT absente ou non finie : {key}")
        indexed[key] = row
    if set(indexed) != {(model, seed) for model in MODELS for seed in SEEDS}:
        raise ValueError("La matrice quatre modèles × sept seeds doit être complète.")
    return payload, indexed, hashlib.sha256(raw).hexdigest()


def _existing_manifest(output_dir: Path, results_sha256: str) -> dict:
    path = output_dir / "figures.json"
    if not path.is_file():
        raise FileExistsError(
            f"Dossier de figures déjà présent sans manifeste complet : {output_dir}"
        )
    manifest = json.loads(path.read_text(encoding="utf-8"))
    if manifest.get("results_sha256") != results_sha256:
        raise ValueError(
            "Les figures existantes ne correspondent pas aux résultats ; choisir un autre output_dir."
        )
    expected = {f"{stem}.{extension}" for stem, _ in FIGURES for extension in ("png", "svg")}
    files = manifest.get("files", [])
    if {Path(item["path"]).name for item in files} != expected:
        raise ValueError("Manifeste de figures incomplet.")
    for item in files:
        candidate = output_dir / Path(item["path"]).name
        if (
            not candidate.is_file()
            or hashlib.sha256(candidate.read_bytes()).hexdigest() != item["sha256"]
        ):
            raise ValueError(f"Figure absente ou modifiée : {candidate}")
    return manifest


def render_plots(run_dir: str | Path, output_dir: str | Path | None = None) -> dict:
    "Enregistre six figures PNG 400 dpi et SVG sans écraser les fichiers existants."

    run_dir = Path(run_dir).expanduser().resolve()
    payload, indexed, result_sha = _read_complete_results(run_dir)
    destination = (
        Path(output_dir).expanduser().resolve() if output_dir is not None else run_dir / "figures"
    )
    if destination.exists():
        return _existing_manifest(destination, result_sha)
    import matplotlib

    matplotlib.use("Agg", force=True)
    import matplotlib.pyplot as plt

    destination.mkdir(parents=True, exist_ok=False)
    files = []
    seed_offsets = tuple((index - 3) * 0.035 for index in range(7))
    caveat = "Seeds de développement sélectionnées ; expérience appariée, sans confirmation indépendante."
    with plt.rc_context(
        {
            "font.family": "DejaVu Sans",
            "font.size": 10,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "svg.fonttype": "none",
        }
    ):

        def finish(figure, stem, title):
            figure.suptitle(title, fontsize=13, fontweight="bold")
            figure.text(0.01, 0.015, caveat, fontsize=8, color="#555555")
            figure.tight_layout(rect=(0, 0.065, 1, 0.94))
            for extension in ("png", "svg"):
                path = destination / f"{stem}.{extension}"
                temporary = destination / f".{stem}.{extension}.part"
                figure.savefig(
                    temporary, format=extension, dpi=400, facecolor="white", bbox_inches="tight"
                )
                os.replace(temporary, path)
                files.append(
                    {
                        "path": str(path),
                        "format": extension,
                        "dpi": 400 if extension == "png" else None,
                        "title": title,
                        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
                    }
                )
            plt.close(figure)

        figure, axis = plt.subplots(figsize=(10.8, 5.1))
        for model_index, model in enumerate(MODELS):
            values = [
                [indexed[(model, seed)]["stage_points"][stage]["lm_ce"] for stage in PLOT_STAGES]
                for seed in SEEDS
            ]
            offset = (model_index - 1.5) * 0.07
            for seed_index, series in enumerate(values):
                axis.scatter(
                    [position + offset + seed_offsets[seed_index] / 2 for position in range(3)],
                    series,
                    s=17,
                    color=COLORS[model],
                    alpha=0.42,
                )
            axis.plot(
                [position + offset for position in range(3)],
                [mean(series[position] for series in values) for position in range(3)],
                marker="o",
                linewidth=2,
                color=COLORS[model],
                label=LABELS[model],
            )
        axis.set(
            xticks=range(3),
            xticklabels=("Après préentraînement", "Après raffinement", "Après SFT"),
            ylabel="CE texte (plus faible = meilleur)",
        )
        axis.grid(axis="y", alpha=0.2)
        axis.legend(fontsize=9)
        finish(
            figure, "lm_checkpoints", "Texte : trois checkpoints de validation réellement mesurés"
        )

        def metric_points(metric, ylabel, stem, title):
            figure, axis = plt.subplots(figsize=(9.4, 4.8))
            for position, model in enumerate(MODELS):
                values = [indexed[(model, seed)]["metrics"][metric] for seed in SEEDS]
                axis.scatter(
                    [position + offset for offset in seed_offsets],
                    values,
                    color=COLORS[model],
                    s=35,
                    alpha=0.8,
                )
                axis.plot(
                    (position - 0.18, position + 0.18),
                    (mean(values), mean(values)),
                    color=COLORS[model],
                    linewidth=3,
                )
            axis.set(
                xticks=range(4),
                xticklabels=(
                    "TR-HASH\ncandidat",
                    "Dense E\nattention simple",
                    "Dense C\ntêtes originales",
                    "Dense D\nmêmes têtes et V0",
                ),
                ylabel=ylabel,
            )
            axis.grid(axis="y", alpha=0.2)
            finish(figure, stem, title + " — points : sept seeds ; trait : moyenne")

        metric_points(
            "reponses_finales",
            "CE réponses, validation pure (plus faible = meilleur)",
            "final_responses",
            "Réponses après SFT",
        )

        figure, axis = plt.subplots(figsize=(10.8, 5.1))
        for model in MODELS:
            values = []
            for seed in SEEDS:
                epochs = sorted(
                    (
                        epoch
                        for epoch in indexed[(model, seed)]["epochs"]
                        if epoch["stage"] == "sft"
                    ),
                    key=lambda epoch: epoch["epoch"],
                )
                series = [epoch["train_ce"] for epoch in epochs]
                values.append(series)
                axis.plot(range(1, 13), series, color=COLORS[model], linewidth=0.7, alpha=0.18)
            axis.plot(
                range(1, 13),
                [mean(series[position] for series in values) for position in range(12)],
                marker="o",
                markersize=3,
                linewidth=2,
                color=COLORS[model],
                label=LABELS[model],
            )
        axis.set(
            xlabel="Époque SFT",
            ylabel="CE d’entraînement",
            xticks=range(1, 13),
        )
        axis.grid(alpha=0.2)
        axis.legend(fontsize=9)
        finish(
            figure,
            "sft_training_ce",
            "SFT : douze CE d’entraînement ; aucune validation intermédiaire inventée",
        )

        metric_points(
            "oubli",
            "Variation de CE texte définie dans le protocole (plus faible = meilleur)",
            "forgetting",
            "Oubli après SFT",
        )

        figure, axes = plt.subplots(1, 5, figsize=(15.5, 4.9))
        for axis, metric, label in zip(axes, METRICS, METRIC_LABELS):
            for position, model in enumerate(MODELS[1:]):
                deltas = [
                    indexed[(model, seed)]["metrics"][metric]
                    - indexed[("tr_candidate", seed)]["metrics"][metric]
                    for seed in SEEDS
                ]
                axis.scatter(
                    [position + offset for offset in seed_offsets],
                    deltas,
                    color=COLORS[model],
                    s=23,
                    alpha=0.8,
                )
                axis.plot(
                    (position - 0.19, position + 0.19),
                    (mean(deltas), mean(deltas)),
                    color=COLORS[model],
                    linewidth=2.5,
                )
            axis.axhline(0, color="#333333", linewidth=0.8, linestyle="--")
            axis.set(
                title=label.replace(" après ", "\naprès "),
                xticks=range(3),
                xticklabels=("E", "C", "D"),
            )
            axis.grid(axis="y", alpha=0.18)
        axes[0].set_ylabel("CE contrôle − CE TR-HASH")
        finish(
            figure, "paired_deltas", "Écarts appariés par seed : au-dessus de zéro favorise TR-HASH"
        )

        figure, axis = plt.subplots(figsize=(9.1, 5.7))
        for model in MODELS:
            x_values = [indexed[(model, seed)]["metrics"]["lm_final"] for seed in SEEDS]
            y_values = [indexed[(model, seed)]["metrics"]["reponses_finales"] for seed in SEEDS]
            axis.scatter(
                x_values, y_values, color=COLORS[model], s=34, alpha=0.75, label=LABELS[model]
            )
            axis.scatter(
                [mean(x_values)],
                [mean(y_values)],
                color=COLORS[model],
                marker="D",
                s=65,
                edgecolors="white",
                linewidths=0.8,
            )
            for seed, x_value, y_value in zip(SEEDS, x_values, y_values):
                axis.annotate(
                    str(seed),
                    (x_value, y_value),
                    xytext=(3, 3),
                    textcoords="offset points",
                    fontsize=7,
                    alpha=0.75,
                )
        axis.set(
            xlabel="CE texte finale (plus faible = meilleur)",
            ylabel="CE réponses finale (plus faible = meilleur)",
        )
        axis.grid(alpha=0.2)
        axis.legend(fontsize=9)
        finish(
            figure,
            "final_text_vs_responses",
            "Texte final et réponses finales — losanges : moyennes",
        )

    manifest = {
        "format_version": 1,
        "results_sha256": result_sha,
        "status": "complete",
        "development_only": True,
        "trajectories": len(payload["trajectories"]),
        "seeds": list(SEEDS),
        "files": files,
    }
    temporary = destination / ".figures.json.part"
    temporary.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    os.replace(temporary, destination / "figures.json")
    return manifest


## 6. Lancer et comparer

La première cellule entraîne les quatre modèles depuis zéro ; la suivante affiche les 28 résultats, les graphiques et toutes les comparaisons. Une victoire exige `CE TR − CE contrôle <= -0,001`, sur chacune des cinq métriques.

E et C donnent 70 comparaisons ; D en donne 35. Les sept seeds ont servi au développement : même un succès sur les 105 comparaisons demanderait encore une confirmation indépendante. Aucun résultat partiel ne sert à choisir un gagnant.


In [ ]:
# research_cell:r-run
if RUN_RESEARCH:
    RUN_DIRECTORY = prepare_run()
    print("Nouveau dossier :", RUN_DIRECTORY)
    research_result = run_research(RUN_DIRECTORY)
else:
    print("Entraînement désactivé ; renseigner RUN_DIRECTORY pour relire un résultat.")


In [ ]:
# research_cell:r-results
if RUN_DIRECTORY is not None:
    import pandas as pd
    from IPython.display import display, Image

    RUN_DIRECTORY = Path(RUN_DIRECTORY).expanduser().resolve()
    saved_results = load_saved_results(RUN_DIRECTORY)
    comparison = comparer_resultats(saved_results)
    _atomic_json(RUN_DIRECTORY / "comparison.json", comparison)
    rows = [
        {
            "seed": row["seed"],
            "modèle": row["model"],
            "paramètres": row["num_parameters"],
            **row["metrics"],
        }
        for row in saved_results["trajectories"]
    ]
    display(pd.DataFrame(rows))
    print("Formule de têtes :", saved_results["protocol"]["head1_formula"])
    print("Victoires par contrôle /35 :", comparison["scores_sur_35"])
    print(
        "E + C /70 :",
        comparison["comparaison_E_C_sur_70"],
        "· total /105 :",
        comparison["total_sur_105"],
    )
    failures = [row for row in comparison["grille_complete"] if not row["victoire"]]
    if failures:
        display(pd.DataFrame(failures))
    print("Ces résultats de développement ne constituent pas une confirmation indépendante.")
    figures = render_plots(RUN_DIRECTORY)
    for figure in figures["files"]:
        if figure["format"] == "png":
            print(figure["title"])
            display(Image(filename=figure["path"]))
else:
    print("Aucun dossier de résultats sélectionné.")
